In [34]:
#!printenv
!python3 ../src/preprocess.py \
    --input ../data/raw/output_http_csic_2010_weka_with_duplications_RAW-RFC2616_escd_v02_full.csv \
    --output ../data/processed/csic_2010_processed.csv

[preprocess] Loading raw CSV from: ../data/raw/output_http_csic_2010_weka_with_duplications_RAW-RFC2616_escd_v02_full.csv
[preprocess] Normalizing column names...
[preprocess] Deriving source partitions...
[preprocess] Creating request_id...
[preprocess] Validating required columns...
[preprocess] Validating grouped column consistency...
[preprocess] Aggregating by source-aware request key...
[preprocess] Filling selected missing values...
[preprocess] Validating output invariants...
[preprocess] Raw shape: (223585, 20)
[preprocess] Aggregated shape: (61065, 23)
[preprocess] Unique request count: 61065
[preprocess] Source partitions:
anom    119585
norm    104000
Name: source_partition, dtype: int64
[preprocess] Label distribution:
norm    36000
anom    25065
Name: label, dtype: int64
[preprocess] Group consistency check passed.
[preprocess] Null counts after fill:
content_length    0
content_type      0
payload           0
dtype: int64
[preprocess] Missingness indicator sums:
has_cont

Notebook to validate the processed CSV

In [2]:
from pathlib import Path
import pandas as pd

In [3]:
!ls -l ../data/processed

total 30024
-rw-r--r--. 1 jovyan root 30742216 Aug 11 06:41 csic_2010_processed.csv


In [11]:
# load the processed CSV
PROCESSED_CSV = Path("../data/processed/csic_2010_processed.csv")
df = pd.read_csv(PROCESSED_CSV, low_memory=False)
#print(len(df))
print(f"Shape: {df.shape}")
#df.head()

Shape: (61065, 23)


In [27]:
PROCESSED_CSV = Path("../data/processed/csic_2010_processed.csv")
df = pd.read_csv(PROCESSED_CSV, low_memory=False)

# Fill empty-string columns that pandas reads back as NaN
df["content_type"] = df["content_type"].fillna("")
df["payload"] = df["payload"].fillna("")

print("Shape:", df.shape)
print("Unique request_id:", df["request_id"].nunique())
print("Duplicate request_id:", df["request_id"].duplicated().sum())
print()
print("Null counts:")
print(df[["content_length", "content_type", "payload"]].isna().sum())
print()
print("Missingness indicators:")
print(df[["has_content_length", "has_content_type", "has_payload"]].value_counts())
ok = (
    df["request_id"].nunique() == len(df)
    and df["request_id"].duplicated().sum() == 0
    and set(df["label"].unique()) == {"anom", "norm"}
    and df[["content_length", "content_type", "payload"]].isna().sum().sum() == 0
)

print("Preprocessed CSV ready for featurise:", ok)

Shape: (61065, 23)
Unique request_id: 61065
Duplicate request_id: 0

Null counts:
content_length    0
content_type      0
payload           0
dtype: int64

Missingness indicators:
has_content_length  has_content_type  has_payload
1                   1                 1              35574
                                      0              25491
dtype: int64
Preprocessed CSV ready for featurise: True


In [29]:
print("Shape:", df.shape)
print("Unique request_id:", df["request_id"].nunique())
print("Duplicate request_id:", df["request_id"].duplicated().sum())
print()
print("Null counts:")
print(df[["content_length", "content_type", "payload"]].isna().sum())
print()
print("Missingness indicators:")
print(df[["has_content_length", "has_content_type", "has_payload"]].value_counts())
ok = (
    df["request_id"].nunique() == len(df)
    and df["request_id"].duplicated().sum() == 0
    and set(df["label"].unique()) == {"anom", "norm"}
    and df[["content_length", "content_type", "payload"]].isna().sum().sum() == 0
)

print("Preprocessed CSV ready for featurise:", ok)

Shape: (61065, 23)
Unique request_id: 61065
Duplicate request_id: 0

Null counts:
content_length    0
content_type      0
payload           0
dtype: int64

Missingness indicators:
has_content_length  has_content_type  has_payload
1                   1                 1              35574
                                      0              25491
dtype: int64
Preprocessed CSV ready for featurise: True


In [12]:
print(df.columns)

Index(['source_partition', 'index', 'request_id', 'method', 'url', 'protocol',
       'user_agent', 'pragma', 'cache_control', 'accept', 'accept_encoding',
       'accept_charset', 'accept_language', 'host', 'connection', 'cookie',
       'label', 'content_length', 'content_type', 'payload',
       'has_content_length', 'has_content_type', 'has_payload'],
      dtype='object')


In [13]:
print(df.dtypes)

source_partition       object
index                   int64
request_id             object
method                 object
url                    object
protocol               object
user_agent             object
pragma                 object
cache_control          object
accept                 object
accept_encoding        object
accept_charset         object
accept_language        object
host                   object
connection             object
cookie                 object
label                  object
content_length        float64
content_type           object
payload                object
has_content_length      int64
has_content_type        int64
has_payload             int64
dtype: object


In [6]:
# check unique request count
print("Unique request_id:", df["request_id"].nunique())
print("Duplicate request_id:", df["request_id"].duplicated().sum())

Unique request_id: 61065
Duplicate request_id: 0


In [14]:
# inspect source partitions
df["source_partition"].value_counts()

norm    36000
anom    25065
Name: source_partition, dtype: int64

In [15]:
# inspect label distribution
df["label"].value_counts()

norm    36000
anom    25065
Name: label, dtype: int64

In [17]:
# verify that each request has exactly one label
df.groupby("request_id")["label"].nunique().value_counts()

1    61065
Name: label, dtype: int64

In [18]:
print("Null counts:")
print(df[["content_length", "content_type", "payload"]].isna().sum())
print()

Null counts:
content_length        0
content_type      43088
payload           25491
dtype: int64



In [21]:
print("Missingness indicators:")
print(df[["has_content_length", "has_content_type", "has_payload"]].value_counts())

Missingness indicators:
has_content_length  has_content_type  has_payload
1                   1                 1              35574
                                      0              25491
dtype: int64


In [30]:
# final decision cell
ok = (
    df["request_id"].nunique() == len(df)
    and df["request_id"].duplicated().sum() == 0
    and set(df["label"].unique()) == {"anom", "norm"}
    and df[["content_length", "content_type", "payload"]].isna().sum().sum() == 0
)

print("Preprocessed CSV ready for featurise:", ok)

Preprocessed CSV ready for featurise: True


**Summary:**  
61,065 unique requests (one row per logical HTTP request)  
No duplicate request_id  
Labels: only anom and norm  
Null counts: zeros for missing `content_length`, `content_type`, `payload`  
Missingness indicators:  
- has_content_length = 1 for all rows  
- has_content_type = 1 for all rows  
- has_payload = 1 for 35,574 requests, 0 for 25,491 requests  